# Session 3: MilkLab° Gelato RAG Retrieval Evaluation (Precision@3 & Recall@3)

สมุดบันทึกนี้ประเมินประสิทธิภาพของระบบ Retrieval ของ MilkLab° Gelato RAG Chatbot ด้วย 10 ชุดคำถามภาษาไทย พร้อมวัดค่า Precision@3 และ Recall@3 และแสดงแจกแจง Similarity Score

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from app import load_index

# โหลดโมเดลและ FAISS Index จาก app.py
model, index, chunks = load_index()
print(f"โหลด Chunks ทั้งหมด {len(chunks)} รายการสำเร็จ")

In [ ]:
# 1. เตรียม Ground Truth (10 คำถามไอศกรีมเจลาโต้)
EVAL_DATASET = [
    {
        "query": "ร้านเปิดกี่โมงและปิดกี่โมง",
        "ground_truth": ["[เกี่ยวกับร้าน] MilkLab° Gelato เป็นร้านไอศกรีมเจลาโต้โฮมเมดพรีเมียม เปิดบริการทุกวันยกเว้นวันจันทร์ เวลา 16:00 ถึง 23:00 น."]
    },
    {
        "query": "เจลาโต้นมสดฮอกไกโดราคาเท่าไหร่และขนาดเท่าไหร่",
        "ground_truth": ["[เมนูเจลาโต้หลัก] เจลาโต้นมสดฮอกไกโด (Hokkaido Milk Gelato): 80 บาท (นมสดฮอกไกโดแท้ 100% เข้มข้น หอมนุ่ม) ขนาดถ้วย 120g"]
    },
    {
        "query": "เจลาโต้ดาร์กช็อกโกแลตรสชาติยังไงราคาเท่าไหร่",
        "ground_truth": ["[เมนูเจลาโต้หลัก] เจลาโต้ดาร์กช็อกโกแลต (Dark Chocolate Gelato): 85 บาท (ผงโกโก้พรีเมียมเข้มข้น 70%) ขนาดถ้วย 120g"]
    },
    {
        "query": "มีส่วนผสมของถั่วไหม ลูกค้าแพ้ถั่วกินได้ไหม",
        "ground_truth": ["[Allergen & สารก่อภูมิแพ้] ปลอดถั่ว (Nut-Free ทุกเมนู ลูกค้าแพ้ถั่วทานได้อย่างปลอดภัย)"]
    },
    {
        "query": "มีเจลาโต้รสไหนบ้างที่คนทานมังสวิรัติหรือวีแกนกินได้",
        "ground_truth": [
            "[เมนูเจลาโต้หลัก] เจลาโต้สตรอว์เบอร์รีซอร์เบต์ (Strawberry Sorbet Gelato): 85 บาท (สตรอว์เบอร์รีสดแท้ 100%) ขนาดถ้วย 120g",
            "[Allergen & สารก่อภูมิแพ้] ซอร์เบต์ (สตรอว์เบอร์รีซอร์เบต์, มะม่วงซอร์เบต์): ปลอด dairy/lactose 100% (ผู้ทานมังสวิรัติ / Vegan ทานได้)"
        ]
    },
    {
        "query": "ค่าส่งเท่าไหร่และส่งไกลแค่ไหนมีแพ็คเจลเย็นไหม",
        "ground_truth": ["[FAQ] **ส่งได้ไกลแค่ไหน**: บริการจัดส่งรัศมี 5 กม. พร้อมแพ็คเจลเก็บความเย็น ค่าส่ง 30 บาท"]
    },
    {
        "query": "เจลาโต้ชาเขียวมัทฉะใช้ชาเขียวจากไหนราคาเท่าไหร่",
        "ground_truth": ["[เมนูเจลาโต้หลัก] เจลาโต้ชาเขียวมัทฉะ (Matcha Green Tea Gelato): 90 บาท (ผงมัทฉะเกรดพิธีการจากอุจิ เกียวโต) ขนาดถ้วย 120g"]
    },
    {
        "query": "สั่งจองเจลาโต้ล่วงหน้าทางไหนสั่งได้ถึงกี่โมง",
        "ground_truth": ["[FAQ] **จองล่วงหน้าได้ไหม**: ได้ ผ่าน LINE OA สั่งก่อน 15:00 น."]
    },
    {
        "query": "การสั่งซื้อไอติมเจลาโต้มีกำหนดออเดอร์ขั้นต่ำหรือไม่",
        "ground_truth": ["[FAQ] **ออเดอร์ขั้นต่ำ**: ไม่มีออเดอร์ขั้นต่ำ"]
    },
    {
        "query": "เจลาโต้มะม่วงมหาชนกซอร์เบต์ราคาเท่าไหร่",
        "ground_truth": ["[เมนูเจลาโต้หลัก] เจลาโต้มะม่วงมหาชนกซอร์เบต์ (Mahachanok Mango Sorbet): 80 บาท (เนื้อมะม่วงมหาชนกสดหวานฉ่ำ) ขนาดถ้วย 120g"]
    }
]

In [ ]:
# 2. รัน Retrieval Top-k (k=3) และคำนวณ Precision@3 และ Recall@3
k = 3
precisions = []
recalls = []
top1_scores = []

for i, item in enumerate(EVAL_DATASET, 1):
    query = item["query"]
    gt = item["ground_truth"]
    
    q_emb = model.encode([query], convert_to_numpy=True, normalize_embeddings=True).astype(np.float32)
    scores, indices = index.search(q_emb, k)
    
    retrieved_chunks = [chunks[idx] for idx in indices[0] if 0 <= idx < len(chunks)]
    top1_score = float(scores[0][0])
    top1_scores.append(top1_score)
    
    hits = sum(1 for c in retrieved_chunks if c in gt)
    precision = hits / k
    recall = hits / len(gt) if len(gt) > 0 else 0.0
    
    precisions.append(precision)
    recalls.append(recall)
    
    print(f"[Case {i}] Query: '{query}'")
    print(f"   Hits: {hits}/{len(gt)} | Precision@{k}: {precision:.4f} | Recall@{k}: {recall:.4f} | Top-1 Score: {top1_score:.4f}")

mean_precision = np.mean(precisions)
mean_recall = np.mean(recalls)

print("\n" + "="*60)
print(f"Overall Mean Precision@{k}: {mean_precision:.4f} ({mean_precision*100:.2f}%)")
print(f"Overall Mean Recall@{k}:    {mean_recall:.4f} ({mean_recall*100:.2f}%)")
print(f"Overall Mean Top-1 Score: {np.mean(top1_scores):.4f}")
print("="*60)

In [ ]:
# 3. พล็อต Histogram แจกแจงความถี่ของ Top-1 Similarity Score
plt.figure(figsize=(8, 5))
plt.hist(top1_scores, bins=5, color='teal', edgecolor='black', alpha=0.7)
plt.title('Distribution of Top-1 Similarity Scores (MilkLab Gelato RAG)')
plt.xlabel('Cosine Similarity Score')
plt.ylabel('Frequency')
plt.grid(True, linestyle='--', alpha=0.5)
plt.show()